# A midpoint is not the answer

A midpoint is a rational sitting in the middle of an interval. An interval of samples is a finite list. Neither object evaluates every query you might later ask. Completing the rationals fills gaps; it does not turn a sample list into an arbitrary continuum query. That last obstruction is CS / CS–Q, cited here, not re-proved.

This notebook uses the shipped calculator. Arithmetic is exact (fractions, not floating point).

Run from the package root, or from this `jupyter_notebooks/` folder.

In [1]:
from pathlib import Path
import sys

p = Path.cwd().resolve()
if p.name == "jupyter_notebooks":
    root = p.parent
elif (p / "code" / "coda_q_core.py").is_file():
    root = p
elif (p / "coda_atlas" / "F005_coda" / "q" / "code").is_dir():
    root = p / "coda_atlas" / "F005_coda" / "q"
elif (p / "q" / "code").is_dir():
    root = p / "q"
else:
    root = p
assert (root / "code").is_dir(), "run from the package root or jupyter_notebooks/"
if str(root / "code") not in sys.path:
    sys.path.insert(0, str(root / "code"))

from fractions import Fraction
from coda_q_core import (
    SampledSignal,
    enclosure_error,
    query_collides_on_profile,
    reproduce_sqrt2_picture,
    two_signal_sample_collision,
)

## After five squeezes, the midpoint is still not $\sqrt{2}$

The half-width is $1/64$. Every point still in the interval, including $\sqrt{2}$, is at most that far from the midpoint. The midpoint itself is a rational whose square is not $2$.

In [2]:
pic = reproduce_sqrt2_picture(5)
print("interval:", pic["lower"], "to", pic["upper"])
print("midpoint:", pic["midpoint"])
print("half-width:", pic["error_bound"])
print("midpoint squared:", pic["midpoint"] ** 2)
assert pic["verifies"] is True
assert pic["error_bound"] == Fraction(1, 64)
assert pic["midpoint"] ** 2 != 2
assert enclosure_error(pic["lower"], pic["upper"], pic["midpoint"]) == 0
assert enclosure_error(pic["lower"], pic["upper"], pic["lower"]) == pic["error_bound"]

interval: 45/32 to 23/16
midpoint: 91/64
half-width: 1/64
midpoint squared: 8281/4096


## Two signals, same two samples

Look only at $t=0$ and $t=1$. The zero function and $g(t)=t(t-1)$ both sample as $(0,0)$. Ask for the value at $1/2$: one returns $0$, the other returns $-1/4$. The sample list does not determine the query. That is the CS four-dots collision in a polynomial costume, and the CS–Q fibre collision on a two-point profile.

In [3]:
report = two_signal_sample_collision()
print("samples on {0,1}:", report["profile"])
print("zero function at 1/2:", report["query_f"])
print("t(t-1) at 1/2:", report["query_g"])
print("collision?", report["collides"])
assert report["profile"] == (Fraction(0), Fraction(0))
assert report["query_f"] == Fraction(0)
assert report["query_g"] == Fraction(-1, 4)
assert report["collides"] is True

samples on {0,1}: (Fraction(0, 1), Fraction(0, 1))
zero function at 1/2: 0
t(t-1) at 1/2: -1/4
collision? True


## When the query already agrees, the samples suffice for that query

Two copies of the zero function agree at $1/2$ as well. The collision test then returns false: this query does not split the fibre. CS–Q says a query that *does* split the fibre cannot factor through the profile. The calculator only checks the finite example.

In [4]:
zero = Fraction(0)
one = Fraction(1)
half = Fraction(1, 2)
f = SampledSignal("f", {zero: zero, one: zero, half: zero})
twin = SampledSignal("twin", {zero: zero, one: zero, half: zero})
print("same samples, same query at 1/2?", not query_collides_on_profile(f, twin, (zero, one), half))
assert query_collides_on_profile(f, twin, (zero, one), half) is False
assert f.evaluate_at(half) == twin.evaluate_at(half)

same samples, same query at 1/2? True
